# 🎙 SoulEcho — GPU Server (Whisper + LLM)

Ce notebook expose une API FastAPI via ngrok avec :
- **STT** : Whisper `large-v3` sur GPU T4 (transcription audio)
- **LLM** : Qwen2.5-7B-Instruct sur GPU T4 (réponses chat streamées)

## Instructions
1. Active le GPU : *Settings → Accelerator → GPU T4 x2*
2. Active Internet : *Settings → Internet → On*
3. Renseigne ton token ngrok dans la cellule de config
4. Lance toutes les cellules dans l'ordre
5. Copie l'URL ngrok affichée et colle-la dans ton `.env` backend : `KAGGLE_GPU_URL=https://xxxx.ngrok-free.app`

In [ ]:
# ═══════════════════════════════════════════════════════
# CONFIGURATION — remplis ces valeurs avant de lancer
# ═══════════════════════════════════════════════════════

NGROK_TOKEN    = ""   # Ton token ngrok : https://dashboard.ngrok.com/get-started/your-authtoken
API_SECRET_KEY = ""   # Clé secrète partagée avec ton backend (génère avec : python -c "import secrets; print(secrets.token_urlsafe(32))")

WHISPER_MODEL  = "large-v3"          # Modèle Whisper
LLM_MODEL_ID   = "Qwen/Qwen2.5-7B-Instruct"  # Modèle LLM HuggingFace
LLM_MAX_TOKENS = 1024

assert NGROK_TOKEN,    "❌ Renseigne NGROK_TOKEN"
assert API_SECRET_KEY, "❌ Renseigne API_SECRET_KEY"
print("✅ Config OK")

In [ ]:
# ═══════════════════════════════════════════════════════
# VÉRIFICATION INTERNET
# Si tu vois une erreur ici → Settings → Internet → On
# ═══════════════════════════════════════════════════════
import urllib.request
try:
    urllib.request.urlopen('https://pypi.org', timeout=5)
    print("✅ Internet OK")
except Exception as e:
    raise RuntimeError(
        "❌ Pas d'accès Internet !\n"
        "→ Dans le panneau droit : Settings → Internet → ON\n"
        "→ Puis relance toutes les cellules"
    ) from e

In [ ]:
# ═══════════════════════════════════════════════════════
# INSTALLATION DES DÉPENDANCES
# ═══════════════════════════════════════════════════════
import subprocess, sys

# Tout en une seule commande pip — plus rapide et plus fiable
packages = [
    "faster-whisper",
    "fastapi",
    "uvicorn[standard]",
    "pyngrok",
    "python-multipart",
    "transformers>=4.40.0",
    "accelerate",
    "bitsandbytes",
    "sse-starlette",
    "av",          # PyAV — conversion webm → wav
    "numpy",
]

print("Installation en cours...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + packages,
    capture_output=True, text=True
)
if result.returncode != 0:
    print(result.stderr[-2000:])
    raise RuntimeError("Installation échouée — vérifie Internet")

print("✅ Dépendances installées")

In [ ]:
# ═══════════════════════════════════════════════════════
# CHARGEMENT DES MODÈLES
# ═══════════════════════════════════════════════════════
import torch
from faster_whisper import WhisperModel
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer
from threading import Thread

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── Whisper ──────────────────────────────────────────
print(f"\nChargement Whisper {WHISPER_MODEL}...")
whisper_compute = "float16" if device == "cuda" else "int8"
whisper_model = WhisperModel(
    WHISPER_MODEL,
    device=device,
    compute_type=whisper_compute,
)
print("✅ Whisper prêt")

# ── LLM ──────────────────────────────────────────────
print(f"\nChargement LLM {LLM_MODEL_ID}...")
llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_ID,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto",
)
llm_model.eval()
print("✅ LLM prêt")
print(f"\nVRAM utilisée: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
# ═══════════════════════════════════════════════════════
# FASTAPI APP
# ═══════════════════════════════════════════════════════
import asyncio
import io
import json
import logging
import os
import tempfile
import wave
from typing import AsyncGenerator, List, Optional

import av
import numpy as np
from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile, status
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from sse_starlette.sse import EventSourceResponse

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("soulecho_gpu")

app = FastAPI(title="SoulEcho GPU Server")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)


# ── Auth middleware ───────────────────────────────────
def verify_key(x_api_key: str = Header(...)):
    if x_api_key != API_SECRET_KEY:
        raise HTTPException(status_code=401, detail="Invalid API key")


# ── Conversion audio → WAV via PyAV ──────────────────
def convert_to_wav(input_path: str, output_path: str) -> bool:
    try:
        with av.open(input_path) as container:
            audio_stream = next((s for s in container.streams if s.type == 'audio'), None)
            if not audio_stream:
                return False
            resampler = av.AudioResampler(format='s16', layout='mono', rate=16000)
            frames = []
            for frame in container.decode(audio_stream):
                for r in resampler.resample(frame):
                    frames.append(r.to_ndarray())
        if not frames:
            return False
        samples = np.concatenate(frames, axis=1).flatten()
        with wave.open(output_path, 'wb') as wf:
            wf.setnchannels(1)
            wf.setsampwidth(2)
            wf.setframerate(16000)
            wf.writeframes(samples.tobytes())
        return True
    except Exception as e:
        logger.warning(f"PyAV conversion failed: {e}")
        return False


# ── STT endpoint ─────────────────────────────────────
@app.post("/transcribe")
async def transcribe(
    file: UploadFile = File(...),
    language: str = Form(default="fr"),
    x_api_key: str = Header(...),
):
    if x_api_key != API_SECRET_KEY:
        raise HTTPException(status_code=401, detail="Invalid API key")

    audio_bytes = await file.read()
    if not audio_bytes:
        raise HTTPException(status_code=400, detail="Empty audio file")

    ext = "webm"
    ct = file.content_type or ""
    if "ogg" in ct: ext = "ogg"
    elif "wav" in ct: ext = "wav"
    elif "mp3" in ct or "mpeg" in ct: ext = "mp3"

    with tempfile.NamedTemporaryFile(suffix=f".{ext}", delete=False) as tmp:
        tmp.write(audio_bytes)
        tmp_path = tmp.name

    wav_path = tmp_path + ".wav"
    try:
        converted = convert_to_wav(tmp_path, wav_path)
        transcribe_path = wav_path if converted else tmp_path

        lang_map = {"fr": "fr", "en": "en", "es": "es", "de": "de", "pt": "pt", "ar": "ar"}
        whisper_lang = lang_map.get(language)

        loop = asyncio.get_event_loop()
        def _transcribe():
            segments, info = whisper_model.transcribe(
                transcribe_path,
                language=whisper_lang,
                beam_size=5,           # GPU → on peut se permettre beam_size=5
                vad_filter=True,
                vad_parameters={"min_silence_duration_ms": 200, "speech_pad_ms": 200},
                log_prob_threshold=-2.0,
                no_speech_threshold=0.95,
            )
            segs = list(segments)
            text = " ".join(s.text.strip() for s in segs).strip()
            if not text:
                # Retry sans VAD
                segments2, _ = whisper_model.transcribe(
                    transcribe_path,
                    language=whisper_lang,
                    beam_size=5,
                    vad_filter=False,
                    log_prob_threshold=-2.0,
                    no_speech_threshold=0.95,
                )
                text = " ".join(s.text.strip() for s in list(segments2)).strip()
            logger.info(f"[STT] '{text[:80]}'")
            return text

        text = await loop.run_in_executor(None, _transcribe)
        return {"text": text, "language": language}

    finally:
        for p in [tmp_path, wav_path]:
            try: os.unlink(p)
            except: pass


# ── LLM Chat endpoint (SSE streaming) ────────────────
class ChatMessage(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    messages: List[ChatMessage]
    max_tokens: Optional[int] = LLM_MAX_TOKENS
    temperature: Optional[float] = 0.7


@app.post("/chat")
async def chat_stream(
    request: ChatRequest,
    x_api_key: str = Header(...),
):
    if x_api_key != API_SECRET_KEY:
        raise HTTPException(status_code=401, detail="Invalid API key")

    async def generate() -> AsyncGenerator[str, None]:
        try:
            # Formater les messages pour le modèle
            msgs = [{"role": m.role, "content": m.content} for m in request.messages]
            input_ids = llm_tokenizer.apply_chat_template(
                msgs,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt",
            ).to(llm_model.device)

            streamer = TextIteratorStreamer(
                llm_tokenizer,
                skip_prompt=True,
                skip_special_tokens=True,
            )

            gen_kwargs = dict(
                input_ids=input_ids,
                streamer=streamer,
                max_new_tokens=request.max_tokens,
                temperature=request.temperature,
                do_sample=request.temperature > 0,
                pad_token_id=llm_tokenizer.eos_token_id,
            )

            # Lancer la génération dans un thread
            thread = Thread(target=llm_model.generate, kwargs=gen_kwargs)
            thread.start()

            # Streamer les tokens au fur et à mesure
            for token in streamer:
                if token:
                    yield json.dumps({"type": "chunk", "content": token})
                await asyncio.sleep(0)  # yield control

            thread.join()
            yield json.dumps({"type": "done", "content": ""})

        except Exception as e:
            logger.error(f"[LLM] Error: {e}")
            yield json.dumps({"type": "error", "content": str(e)})

    return EventSourceResponse(generate(), media_type="text/event-stream")


# ── Health check ──────────────────────────────────────
@app.get("/health")
async def health():
    return {
        "status": "ok",
        "whisper": WHISPER_MODEL,
        "llm": LLM_MODEL_ID,
        "device": device,
        "vram_used_gb": round(torch.cuda.memory_allocated() / 1e9, 2) if device == "cuda" else 0,
    }

print("✅ FastAPI app définie")

In [ ]:
# ═══════════════════════════════════════════════════════
# DÉMARRAGE DU SERVEUR + TUNNEL NGROK
# ═══════════════════════════════════════════════════════
import threading
import time
import uvicorn
from pyngrok import ngrok, conf

# Configurer ngrok
conf.get_default().auth_token = NGROK_TOKEN

# Démarrer uvicorn dans un thread
PORT = 8080

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)  # Attendre que le serveur démarre

# Ouvrir le tunnel ngrok
tunnel = ngrok.connect(PORT, "http")
public_url = tunnel.public_url

print("\n" + "="*60)
print("🚀 SOULECHO GPU SERVER EN LIGNE")
print("="*60)
print(f"\n📡 URL publique : {public_url}")
print(f"\n🔑 Ajoute dans ton .env backend :")
print(f"   KAGGLE_GPU_URL={public_url}")
print(f"   KAGGLE_API_KEY={API_SECRET_KEY}")
print(f"\n🏥 Health check : {public_url}/health")
print("\n" + "="*60)
print("⚠️  Garde ce notebook ouvert — la session expire dans 12h")
print("="*60)

In [ ]:
# ═══════════════════════════════════════════════════════
# KEEP ALIVE — empêche Kaggle de couper la session
# Lance cette cellule en dernier et laisse tourner
# ═══════════════════════════════════════════════════════
import time
from datetime import datetime

print("Session active. Ctrl+C pour arrêter.")
start = datetime.now()

while True:
    elapsed = datetime.now() - start
    hours = int(elapsed.total_seconds() // 3600)
    minutes = int((elapsed.total_seconds() % 3600) // 60)
    print(f"\r⏱  Uptime: {hours:02d}h{minutes:02d}m | URL: {public_url}", end="", flush=True)
    time.sleep(30)